# Repartition vs Coalesce: Full Shuffle vs Local Partition Merge

**Module**: 05_partitions_and_shuffle  
**Notebook**: 02_repartition_and_coalesce.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The architectural difference between repartition() and coalesce(), when to use each, and how to avoid the small files problem.
- **Why does this topic exist?**: Using repartition when coalesce suffices triggers a massive, unnecessary network shuffle.
- **What problem does it solve?**: Slow writes and excessive network saturation.
- **Where is it used in real-world Data Engineering?**: Downsizing partition counts after filtering before writing to Parquet data lakes.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Partitioning fundamentals, lazy evaluation |
| **Required depth** | Working knowledge: can choose between repartition and coalesce and explain the cost |
| **Data Engineer** | Resize partitions correctly before writes and joins without adding unnecessary shuffles. |
| **Senior Data Engineer** | Diagnose partition-count problems and choose a layout that serves downstream readers. |
| **Spark specialist** | Reason about partition coalescing, shuffle boundaries, and file-sizing side effects. |
| **Next topic** | `05_partitions_and_shuffle/03_shuffle_internals.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

`repartition` and `coalesce` both change the number of partitions, but they solve different problems.
`repartition` performs a full shuffle and can redistribute data evenly; `coalesce` merges adjacent partitions and
usually avoids a shuffle. Choosing the wrong one creates either a costly network exchange or an uneven output that
becomes the next job's bottleneck.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Partition** | One unit of parallel work | A slice of the dataset processed by one task |
| **repartition** | Reshuffle into a new partition count/layout | Full exchange; can hash by columns or range-partition |
| **coalesce** | Merge adjacent partitions | Narrow dependency by default; optional shuffle flag |
| **Shuffle** | Network redistribution of records | Exchange node in the physical plan |
| **Partition pruning** | Skip irrelevant partitions | Especially valuable for file-source partition columns |
| **Small-file problem** | Many tiny output files | Caused by too many partitions or frequent appends |

### How it works, step by step

1. The current plan reports its partition count and partitioning expression.
2. `repartition(n)` inserts an exchange that hash-distributes rows across `n` partitions.
3. `repartition(n, col)` hashes the given columns so equal keys land together.
4. `coalesce(n)` builds a plan that merges adjacent parent partitions without a network shuffle.
5. The scheduler runs one task per resulting partition; empty or tiny partitions still consume task slots.
6. Writers emit one file per non-empty partition (plus directory overhead), so partition count affects file count.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Architectural Comparison

```
COALESCE(2) [Local Merge - Narrow Transformation, No Shuffle]:
Worker 1: [P0] + [P1] ----> [Merged P0]
Worker 2: [P2] + [P3] ----> [Merged P1]
(No data travels across the network between workers!)

REPARTITION(2) [Round-Robin / Hash - Wide Transformation, Full Shuffle]:
Worker 1: [P0], [P1] ---\ /--- Network Shuffle ---> [New P0]
Worker 2: [P2], [P3] ---/ \--- Network Shuffle ---> [New P1]
(All records serialized and exchanged over network!)
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Repartition_Coalesce_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Benchmarking Repartition vs Coalesce Execution Plans

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

# Start with 8 partitions
base_df = orders.repartition(8)
print("Base partitions:", base_df.rdd.getNumPartitions())

print("\n=== COALESCE(2) Physical Plan (Notice: NO Exchange!) ===")
base_df.coalesce(2).explain()

print("\n=== REPARTITION(2) Physical Plan (Notice: Exchange Node!) ===")
base_df.repartition(2).explain()

---

## Experiment: Shuffle Write Bytes: Coalesce vs Repartition

- **1. What are we testing?**: Compare shuffle metrics between coalesce and repartition.
- **2. What do we expect?**: coalesce writes 0 shuffle bytes; repartition writes 100% of data to shuffle files.

In [ ]:
t0 = time.time()
base_df.coalesce(2).write.mode("overwrite").format("noop").save()
t_coal = time.time() - t0

t0 = time.time()
base_df.repartition(2).write.mode("overwrite").format("noop").save()
t_rep = time.time() - t0

print(f"Coalesce(2) time:    {t_coal:.3f}s (Zero shuffle)")
print(f"Repartition(2) time: {t_rep:.3f}s (Full network shuffle)")

**3. What happened?**: Coalesce completed significantly faster with zero shuffle overhead.  
**4. Why did it happen?**: Coalesce merged partition slices locally on each worker without network transfer.  
**5. Practical Engineering Lesson**: Always use coalesce() when reducing partition count after filters or before writes.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Right-sizing a nightly export

- **Industry**: Retail analytics
- **Scenario**: A nightly job writes a 40 GB curated table to object storage for a BI engine.
- **Problem**: The upstream job leaves 2,000 tiny partitions, producing thousands of small files that slow every reader.
- **Why this concept is relevant**: A single `coalesce` or a `repartition` by the write key can reduce file count; the choice depends on whether even distribution matters.
- **Implementation idea**: Measure partition sizes, coalesce to a target file size, and write with a deterministic layout.
- **Expected result**: Fewer, larger files and lower read latency for the BI layer.
- **Alternative**: A compaction job that rewrites small files later.
- **Trade-offs**: Coalescing is cheap but can leave skew; repartitioning is even but pays a shuffle.

### Use Case 2: Preparing data for a downstream join

- **Industry**: Telecom
- **Scenario**: A call-detail pipeline joins a 500 GB fact table to a subscriber dimension on `subscriber_id`.
- **Problem**: The fact table arrives with arbitrary partitioning, so the join shuffles and then processes highly uneven partitions.
- **Why this concept is relevant**: Repartitioning the fact table by `subscriber_id` before the join can align the layout, but only if the downstream join uses the same key and the extra exchange is reused.
- **Implementation idea**: Compare the plan with and without the pre-repartition; check whether AQE already coalesces the shuffle.
- **Expected result**: A plan with one exchange instead of two, or a decision that the pre-repartition is unnecessary.
- **Alternative**: Let the join's own exchange handle distribution.
- **Trade-offs**: A pre-repartition only pays off when multiple downstream operations reuse the layout.

---

## Business Value

- Lower storage and metadata overhead from small files.
- Faster downstream scans and more predictable SLAs.
- Better cluster utilization by avoiding empty or tiny tasks.
- Clear cost ownership: every extra shuffle has a network and CPU price.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Inspect the physical plan for `Exchange` before and after the change.
- Do not use `coalesce(1)` on large data; it collapses parallelism and can exhaust one executor.
- Prefer column-based `repartition` when a downstream join or aggregation uses the same key.
- Check output file sizes, not only the number of partitions.
- Remember that AQE can coalesce shuffle partitions automatically; avoid fighting it with manual settings.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Measure the cost of repartition versus coalesce

**Steps**

1. Create a DataFrame with an uneven key distribution.
2. Write it with `coalesce(4)` and inspect partition sizes in the UI.
3. Write it with `repartition(4)` and compare shuffle bytes, task durations, and output file sizes.

**What to observe**: Coalesce avoids a shuffle but may leave most rows in one task; repartition spreads rows but pays an exchange.

**Healthy result**: You can state which operation is cheaper for the specific goal.

**Problematic result**: The output has one huge file or the shuffle dominates runtime.

**Engineering takeaway**: Partition count is a means, not a goal; measure the distribution and the downstream read pattern.

---

## Trade-offs and Alternatives

### Advantages

- Simple API
- Immediate control over parallelism
- Can improve file layout and join locality

### Limitations

- Repartition is expensive
- Coalesce can create skew
- Partition counts interact with AQE and writers

### When to use it

- Fixing small files
- Aligning a reused join key
- Reducing task overhead

### When not to use it

- Random repartition calls
- coalesce(1) on large datasets
- Tuning without measuring

### Alternatives inside Spark

- repartition
- repartitionByRange
- coalesce
- AQE coalescing

### Alternatives outside Spark

- Table format compaction
- Managed compaction services
- File-size-aware ingestion

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Coalesce Single Output File
**Problem Statement**: Write sales.csv to Parquet using coalesce(1).  
**Input / Context**: sales.csv

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Hash Repartitioning by Key
**Problem Statement**: Repartition orders by customer_id into 4 partitions and check if rows with same customer reside in same partition.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Detect Coalesce Partition Skew
**Problem Statement**: Create an uneven RDD, coalesce it, and inspect partition distribution using glom.  
**Input / Context**: glom

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The mysterious single-file write

**Scenario**: A pipeline calls `coalesce(1)` before writing a 200 GB table and one task runs for hours.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the write stage's task count and max task duration.
2. Confirm whether the single partition fits on one executor.
3. Replace the call with a size-aware partition count or let AQE coalesce.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Coalesce Single Output File):
```python
spark.read.option('header', True).csv('/opt/spark-data/sales.csv').coalesce(1).write.mode('overwrite').parquet('/opt/spark-data/warehouse/test_single_file')
```

#### Solution 2 (Hash Repartitioning by Key):
```python
p_df = orders.repartition(4,'customer_id')
print('Partitions:', p_df.rdd.getNumPartitions())
```

#### Solution 3 (Detect Coalesce Partition Skew):
```python
s = orders.coalesce(2).rdd.glom().map(len).collect()
print('Coalesced partition distribution:', s)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **coalesce**: Merges locally on node; no shuffle; CAN ONLY DECREASE.
- **repartition**: Full network shuffle; increases or decreases; balances partition sizes.

### Essential Technical Terms
- `Exchange`: Shuffle node in plan.
- `Small Files Problem`: Too many tiny files written to storage.

### Common Engineering Mistakes
- **Using coalesce to increase partitions**: Fails silently; partition count remains unchanged.

### Interview Questions on This Topic
- **Q: When should you use repartition instead of coalesce?**
  - *A*: When increasing partition count, eliminating data skew, or partitioning explicitly by hash keys.


### More Interview Questions

- **Q: When does coalesce avoid a shuffle?**
  - *A*: By default `coalesce` merges adjacent partitions using a narrow dependency, so no network exchange occurs; it can still leave uneven partition sizes.
- **Q: Why is `repartition` sometimes worth the shuffle?**
  - *A*: It redistributes rows evenly or by key, which can reduce skew and make a downstream join or write more efficient.

### What to Study Next
Open 03_shuffle_internals.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')